# Milestone 2 lab, notebook 4 of 8: Rules of the business

## Where we are

| Notebook | What you do |
|---|---|
| 1. Empty in which rows | Write a check for empty cells that does not fire on good rows |
| 2. Unique on which columns | Find the receipt and the file that were counted twice |
| 3. A believable value | Catch 1,180 kg of bananas without catching 400 real bottles |
| **4. Rules of the business** (you are here) | Three rules a person must tell you. Each needs a join first |
| 5. Across two tables | Hold the receipts against the tills' own totals. Find the lines a join dropped |
| 6. A normal day | Find the store that sent nothing. Work out what a normal day is |
| 7. The false alarm | A holiday is not an error. See what silence costs |
| 8. The production version | The same checks as a tested program, and what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import bonus, keep, lab_copy, money, plant, prepare_week, production, sql, true_net_sales

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 24,384 sales lines.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: A rule nobody wrote down

The checks so far looked at the data and asked: is it empty, is it twice, is it the right size and shape? You could write them without knowing what Lotus sells.

Now think back to **Tomas Reyes** and store `S03`. His refund of $9.49 was stored as a positive amount, and it made his Monday $18.98 too high. What rule did that row break?

*A refund gives money back.*

No column says that. You know it because you know what a shop is. A rule of how the company works is called a **business rule**. The data cannot tell you the rule. A person must.

Write it as a check.

In [2]:
GIVES_MONEY_BACK = """
    SELECT line_id, store_id, sales_date, line_type, quantity, amount_cents
    FROM clean.sales_lines
    WHERE (line_type = 'refund' AND (amount_cents >= 0 OR quantity >= 0))
       OR (line_type = 'sale' AND (amount_cents <= 0 OR quantity <= 0))
"""
yours = sql(GIVES_MONEY_BACK)
yours

,line_id,store_id,sales_date,line_type,quantity,amount_cents
0,S03-0824-000008,S03,2026-08-24,refund,-1,949


In [3]:
keep("refund_gives_money_back", yours)

Your query:           1 bad rows
The production check: 1 bad rows  (A refund gives money back)
The same rows.


One row, in five weeks of data. The rule has two halves, and you need both: a refund is below zero, **and** a sale is above zero.

## Step 2: A refund is not larger than the sale

There is a second rule about refunds. You cannot give back more than the customer paid.

To test it, one row is not enough. You need the refund **and** the receipt it refunds. Look at Tomas's refund next to its sale.

In [4]:
sql("""
    SELECT receipt_id, sales_date, line_type, product_name, quantity, amount_cents, refund_of_receipt_id
    FROM clean.sales_lines
    WHERE receipt_id IN ('S03-T1-004493', 'S03-T2-004457')
    ORDER BY sales_date, line_no
""")

,receipt_id,sales_date,line_type,product_name,quantity,amount_cents,refund_of_receipt_id
0,S03-T2-004457,2026-08-22,sale,Olive oil 500ml,1,949,NaN
1,S03-T2-004457,2026-08-22,sale,Sourdough loaf,1,525,NaN
2,S03-T1-004493,2026-08-24,refund,Olive oil 500ml,-1,949,S03-T2-004457


The olive oil was sold on Saturday for 949, on receipt `S03-T2-004457`. It came back on Monday, and the refund names that receipt.

So the check must **join** each refund to the receipt it names, and compare the two amounts. It fails in two ways: the money given back is more than was paid, or the receipt does not exist at all.

In [5]:
NOT_LARGER = """
    SELECT r.line_id, r.store_id, r.sales_date, r.refund_of_receipt_id,
           abs(r.amount_cents) AS given_back_cents, sold.amount_cents AS sold_cents
    FROM clean.sales_lines r
    LEFT JOIN (
        SELECT receipt_id, product_id, sum(amount_cents) AS amount_cents
        FROM clean.sales_lines
        WHERE line_type = 'sale'
        GROUP BY receipt_id, product_id
    ) sold
      ON sold.receipt_id = r.refund_of_receipt_id AND sold.product_id = r.product_id
    WHERE r.line_type = 'refund'
      AND (sold.receipt_id IS NULL OR abs(r.amount_cents) > sold.amount_cents)
"""
yours = sql(NOT_LARGER)
keep("refund_not_larger_than_sale", yours)

Your query:           0 bad rows
The production check: 0 bad rows  (A refund is not larger than the sale)
The same rows.


Zero rows. All 89 refunds give back what was paid, or less.

See it fail once. In the private copy, one refund gives back three times the money.

In [6]:
lab_copy()
plant("""
    UPDATE lab.sales_lines SET amount_cents = amount_cents * 3
    WHERE line_id = (SELECT min(line_id) FROM lab.sales_lines WHERE line_type = 'refund')
""")
print("A refund gives money back:", len(production("refund_gives_money_back", on_the_lab=True)), "bad row. Still only Tomas's refund.")
production("refund_not_larger_than_sale", on_the_lab=True)

lab.sales_lines is a fresh copy of clean.sales_lines: 24,384 rows.
Planted in lab.sales_lines.
A refund gives money back: 1 bad row. Still only Tomas's refund.


,line_id,store_id,sales_date,refund_of_receipt_id,given_back_cents,sold_cents
0,S01-0729-000053,S01,2026-07-29,S01-T1-004010,687,229


The first refund check says nothing new: that refund still has the right sign. Only the check with the join sees it. Two rules, two checks.

## Step 3: The price of the day

**Ines Marchetti** manages store `S04`. On Thursday the buying team typed a new price for butter. Here is the price list for that product.

In [7]:
sql("""
    SELECT product_id, product_name, unit_price_cents, valid_from, valid_to
    FROM clean.products
    WHERE product_id = 'P-003'
    ORDER BY valid_from
""")

,product_id,product_name,unit_price_cents,valid_from,valid_to
0,P-003,Butter 250g,349,2026-01-05,2026-08-27
1,P-003,Butter 250g,399,2026-08-28,NaN


Butter is 349 until Thursday 27 August, and 399 from Friday 28 August. Now look at what each store charged on that Friday.

In [8]:
sql("""
    SELECT store_id, unit_price_cents, sum(quantity) AS packs
    FROM clean.sales_lines
    WHERE product_id = 'P-003' AND sales_date = DATE '2026-08-28'
    GROUP BY store_id, unit_price_cents
    ORDER BY store_id
""")

,store_id,unit_price_cents,packs
0,S01,399,7
1,S02,399,14
2,S03,399,7
3,S04,349,12
4,S05,399,11
5,S06,399,1


Five stores charged 399. Ines's store charged 349, for 12 packs. Its tills still had yesterday's price.

Every one of those lines is a perfectly good line. 349 is a real price for butter. It is just not the price **of that day**.

The check joins each line to the price row that is valid on the day of the sale.

In [9]:
PRICE_OF_THE_DAY = """
    SELECT l.line_id, l.store_id, l.sales_date, l.product_name, l.quantity,
           l.unit_price_cents AS charged_cents, p.unit_price_cents AS price_list_cents,
           CAST(round(l.quantity * (p.unit_price_cents - l.unit_price_cents)) AS BIGINT)
               AS store_short_cents
    FROM clean.sales_lines l
    JOIN clean.products p
      ON p.product_id = l.product_id
     AND l.sales_date BETWEEN p.valid_from AND coalesce(p.valid_to, DATE '9999-12-31')
    WHERE l.unit_price_cents <> p.unit_price_cents
"""
yours = sql(PRICE_OF_THE_DAY)
print("The store is short by", money(yours["store_short_cents"].sum()))
yours

The store is short by $6.00


,line_id,store_id,sales_date,product_name,quantity,charged_cents,price_list_cents,store_short_cents
0,S04-0828-000010,S04,2026-08-28,Butter 250g,2,349,399,100
1,S04-0828-000025,S04,2026-08-28,Butter 250g,2,349,399,100
2,S04-0828-000042,S04,2026-08-28,Butter 250g,2,349,399,100
3,S04-0828-000066,S04,2026-08-28,Butter 250g,2,349,399,100
4,S04-0828-000079,S04,2026-08-28,Butter 250g,2,349,399,100
5,S04-0828-000110,S04,2026-08-28,Butter 250g,2,349,399,100


In [10]:
keep("price_of_the_day", yours)

Your query:           6 bad rows
The production check: 6 bad rows  (The price charged is the price valid that day)
The same rows.


Six lines, and the store took $6.00 less than the price list says.

The price list is **reference data**: a small list that people keep by hand, and that other data is judged against. Prices, opening hours and holidays are all reference data.

Now be careful about who is hurt. Is Ines's net sales number wrong?

In [11]:
reported = int(sql("""
    SELECT net_sales_cents FROM output.daily_store_sales
    WHERE store_id = 'S04' AND sales_date = DATE '2026-08-28'
""").iloc[0, 0])
print("The pipeline says:  ", money(reported))
print("The answer key says:", money(true_net_sales("S04", "2026-08-28")))

The pipeline says:   $844.60
The answer key says: $844.60


They are equal. The table is **right**: the tills really took that money. Ines's bonus is correct.

So this check does not protect pay. It protects the **store's money**: $6.00 that should have been taken and was not. Finance and the store need to know, today, before a second day at the old price. A check can protect something that no output table shows.

This is the difference between two words:

- A **valid** value is allowed by the rules. 349 is a valid price: a whole number of cents, above zero.
- An **accurate** value matches the real world. On 28 August the real price of butter was 399.

A value can be valid and wrong.

## Step 4: Nothing is sold while the store is closed

Back to **Dana Okafor** and store `S02`. Look at the last two days the pipeline has for her.

In [12]:
sql("""
    SELECT sales_date, receipts, lines, net_sales_cents
    FROM output.daily_store_sales
    WHERE store_id = 'S02' AND sales_date >= DATE '2026-08-30'
    ORDER BY sales_date
""")

,sales_date,receipts,lines,net_sales_cents
0,2026-08-30,42,149,88135
1,2026-08-31,1,4,2395


There is a row for **Monday 31 August**. The pipeline has not loaded Monday yet. Where do those four lines come from?

In [13]:
sql("""
    SELECT c.line_id, c.sold_at, c.product_name, c.amount_cents, r._file_name
    FROM clean.sales_lines c
    JOIN raw.sales_lines r ON r.line_id = c.line_id
    WHERE c.store_id = 'S02' AND c.sales_date = DATE '2026-08-31'
    ORDER BY c.line_id
""")

,line_id,sold_at,product_name,amount_cents,_file_name
0,S02-0830-000142,2026-08-31 03:10:00,Coffee beans 250g,899,sales_lines_S02_2026-08-30.csv
1,S02-0830-000143,2026-08-31 03:10:00,Oat milk 1L,698,sales_lines_S02_2026-08-30.csv
2,S02-0830-000144,2026-08-31 03:10:00,"Eggs, 12",519,sales_lines_S02_2026-08-30.csv
3,S02-0830-000145,2026-08-31 03:10:00,Dark chocolate 100g,279,sales_lines_S02_2026-08-30.csv


They came in **Sunday's** file, and they are stamped 03:10 on Monday morning. The store closes at 22:00. A till restarted after a power cut, and its clock was wrong for one receipt.

The pipeline takes the day of a sale from the till's clock. So $23.95 of Sunday's sales landed on Monday: outside the bonus week. Dana's week is $23.95 short, and her bonus $0.24.

The rule: *nothing is sold while the store is closed*. The check needs each store's opening hours. Those are reference data too, in two places: the `stores` table, and a small calendar of days with other hours.

In [14]:
sql("SELECT calendar_date, store_id, kind, opens_at, closes_at, note FROM clean.calendar ORDER BY 1, 2")

,calendar_date,store_id,kind,opens_at,closes_at,note
0,2026-08-26,S01,public_holiday,10:00:00,16:00:00,Founders' Day
1,2026-08-26,S02,public_holiday,10:00:00,16:00:00,Founders' Day
2,2026-08-26,S03,public_holiday,10:00:00,16:00:00,Founders' Day
3,2026-08-26,S04,public_holiday,10:00:00,16:00:00,Founders' Day
4,2026-08-26,S05,public_holiday,10:00:00,16:00:00,Founders' Day
5,2026-08-26,S06,public_holiday,10:00:00,16:00:00,Founders' Day
6,2026-08-27,S01,closed,None,None,Planned refit


On Founders' Day, a public holiday, every store opens from 10:00 to 16:00. The check uses the calendar's hours when a day has them, and the store's hours on every other day.

In [15]:
OPENING_HOURS = """
    SELECT l.line_id, l.store_id, l.sales_date, l.sold_at,
           coalesce(c.opens_at, s.opens_at) AS opens_at,
           coalesce(c.closes_at, s.closes_at) AS closes_at, l.amount_cents
    FROM clean.sales_lines l
    JOIN clean.stores s ON s.store_id = l.store_id
    LEFT JOIN clean.calendar c
      ON c.store_id = l.store_id AND c.calendar_date = l.sales_date
    WHERE CAST(l.sold_at AS TIME) < coalesce(c.opens_at, s.opens_at)
       OR CAST(l.sold_at AS TIME) > coalesce(c.closes_at, s.closes_at)
"""
yours = sql(OPENING_HOURS)
print("Money on the wrong day:", money(yours["amount_cents"].sum()))
yours

Money on the wrong day: $23.95


,line_id,store_id,sales_date,sold_at,opens_at,closes_at,amount_cents
0,S02-0830-000145,S02,2026-08-31,2026-08-31 03:10:00,08:00:00,22:00:00,279
1,S02-0830-000144,S02,2026-08-31,2026-08-31 03:10:00,08:00:00,22:00:00,519
2,S02-0830-000143,S02,2026-08-31,2026-08-31 03:10:00,08:00:00,22:00:00,698
3,S02-0830-000142,S02,2026-08-31,2026-08-31 03:10:00,08:00:00,22:00:00,899


In [16]:
keep("inside_opening_hours", yours)

Your query:           4 bad rows
The production check: 4 bad rows  (Nothing is sold while the store is closed)
The same rows.


Four lines, $23.95, all from one receipt.

## Step 5: Three rules, three joins

| The business rule | What had to be joined first | It found |
|---|---|---|
| A refund gives money back, and not more than was paid | Each refund to the receipt it names | Tomas's refund |
| The price charged is the price valid that day | Each line to the price list, by date | 12 packs of butter at `S04` |
| Nothing is sold while the store is closed | Each line to the store's hours and the calendar | One receipt stamped 03:10 |

None of these is a one-line check on one column. In each, the data had to be **prepared** before the rule could be tested.

## What you now know

- A business rule is a rule of how the company works. The data cannot tell you it. Ask the people who run the shop.
- Most business rules need a join before they can be tested.
- Reference data is a small list kept by hand that other data is judged against: prices, opening hours, holidays.
- A valid value is allowed by the rules. An accurate value matches the real world. A value can be valid and wrong.
- A check can protect something no output shows. The butter check protects the store's money, not the bonus.

**Say it in an interview:** "Business checks came from asking the business, not from the data. One store charged yesterday's butter price for a day: every line was valid, and the store was $6 short. I caught it by joining each sales line to the price row valid on its date. The sales total was correct, so the check protected finance, not the bonus report."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "What is good data?", the questions "Valid?" and "Accurate?".

Next: [notebook 5](05_across_two_tables.ipynb). Each check so far needed you to imagine the fault first. One check does not.